# 04 - Predict & Deploy: Skoring Nasabah dari Data Raw

Notebook ini mensimulasikan pemakaian model di produksi (mis. dipanggil dari
**Risk Agent**): ambil data mentah seorang nasabah **by `application_id`** dari
7 tabel raw di `data/raw/`, bangun fitur dengan pipeline yang sama seperti
`03_model_training.ipynb`, lalu prediksi kelayakan kredit memakai model yang
sudah dilatih & disimpan (`.pkl`).

Contoh nasabah yang diproses: **`APP202602999`** dan **`APP202603000`**.

Model yang dipakai: **`model_logistic_regression_20260823_184525.pkl`**
(beserta `preprocessor_20260823_184525.pkl` dan
`model_meta_logistic_regression_20260823_184525.pkl` dari run training yang sama).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import pandas as pd
import joblib

DATA_DIR = "../data/raw"
MODEL_DIR = "../models"

# Applicant yang mau diprediksi
APPLICATION_IDS = ["APP202602999", "APP202603000"]

# Artifact model yang dipakai untuk deploy (satu set dari run training yang sama)
TIMESTAMP = "20260823_184525"
MODEL_PATH = f"{MODEL_DIR}/model_logistic_regression_{TIMESTAMP}.pkl"
PREPROCESSOR_PATH = f"{MODEL_DIR}/preprocessor_{TIMESTAMP}.pkl"
META_PATH = f"{MODEL_DIR}/model_meta_logistic_regression_{TIMESTAMP}.pkl"

print("Model path       :", MODEL_PATH)
print("Preprocessor path:", PREPROCESSOR_PATH)
print("Meta path        :", META_PATH)

Model path       : ../models/model_logistic_regression_20260823_184525.pkl
Preprocessor path: ../models/preprocessor_20260823_184525.pkl
Meta path        : ../models/model_meta_logistic_regression_20260823_184525.pkl


## 1. Load Artifact Model Tersimpan

In [2]:
model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)
meta = joblib.load(META_PATH)

categorical_cols = meta["categorical_cols"]
numeric_cols = meta["numeric_cols"]
feature_names = meta["feature_names"]
positive_class = meta["positive_class"]  # "Ditolak"

print(f"Model                : {type(model).__name__} ({meta['model_name']})")
print(f"Jumlah fitur kategorikal: {len(categorical_cols)}")
print(f"Jumlah fitur numerik    : {len(numeric_cols)}")
print(f"Total fitur setelah encoding: {len(feature_names)}")
print(f"Positive class (kelas berisiko): {positive_class}")

Model                : LogisticRegression (logistic_regression)
Jumlah fitur kategorikal: 10
Jumlah fitur numerik    : 23
Total fitur setelah encoding: 58
Positive class (kelas berisiko): Ditolak


## 2. Bangun Fitur dari Data Raw untuk Applicant Terpilih

Query 7 tabel raw, filter ke `application_id`/`NIK` yang diminta, lalu jalankan
join + agregasi yang **identik** dengan Tahap 1 di `03_model_training.ipynb`
(sengaja dibuat identik agar skema fitur yang masuk ke `preprocessor` konsisten
dengan yang dipakai saat training — mencegah *training-serving skew*).

In [3]:
profile_full = pd.read_csv(f"{DATA_DIR}/retail_customer_profile.csv")

applicant_profile = profile_full[profile_full["application_id"].isin(APPLICATION_IDS)].copy()
assert len(applicant_profile) == len(APPLICATION_IDS), "Ada application_id yang tidak ditemukan di data raw"

target_niks = applicant_profile["NIK"].tolist()

print(f"Ditemukan {len(applicant_profile)} applicant:")
applicant_profile[["application_id", "NIK", "company_name", "industry", "label"]]

Ditemukan 2 applicant:


,application_id,NIK,company_name,industry,label
2998,APP202602999,3276015311842999,CV Permana Jaya,Perdagangan,Diterima
2999,APP202603000,3275016008853000,UD Susanto Jaya,Transportasi,Diterima


In [4]:
# --- slik_credit_history: agregasi per NIK (hanya untuk NIK target) ---
slik = pd.read_csv(f"{DATA_DIR}/slik_credit_history.csv")
slik = slik[slik["NIK"].isin(target_niks)]

slik_agg = slik.groupby("NIK").agg(
    worst_collectability=("collectability", "max"),
    n_loans=("slik_record_id", "count"),
    total_outstanding=("outstanding_balance", "sum"),
    total_installment=("installment_amount", "sum"),
).reset_index()
slik_agg["has_credit_history"] = 1

print(f"NIK target dengan riwayat SLIK: {slik_agg.shape[0]} / {len(target_niks)}")
slik_agg

NIK target dengan riwayat SLIK: 1 / 2


,NIK,worst_collectability,n_loans,total_outstanding,total_installment,has_credit_history
0,3275016008853000,1,3,184754206,15024240,1


In [5]:
# --- dhn: status_dhn -> biner ---
dhn = pd.read_csv(f"{DATA_DIR}/dhn.csv")
dhn = dhn[dhn["NIK"].isin(target_niks)]

dhn_feat = dhn[["NIK", "status_dhn"]].copy()
dhn_feat["status_dhn_flag"] = (dhn_feat["status_dhn"] == "Ya").astype(int)
dhn_feat = dhn_feat[["NIK", "status_dhn_flag"]]

dhn_feat

,NIK,status_dhn_flag
2998,3276015311842999,0
2999,3275016008853000,0


In [6]:
# --- agunan_atr_bpn ---
agunan = pd.read_csv(f"{DATA_DIR}/agunan_atr_bpn.csv")
agunan = agunan[agunan["NIK"].isin(target_niks)]

agunan_feat = agunan[["NIK", "asset_type", "total_collateral_value", "land_area_m2"]].copy()
agunan_feat

,NIK,asset_type,total_collateral_value,land_area_m2
2998,3276015311842999,Ruko,12612989000,336.1
2999,3275016008853000,Tanah,8995200000,374.8


In [7]:
# --- laporan_keuangan: 2024 vs 2025 ---
fin = pd.read_csv(f"{DATA_DIR}/laporan_keuangan.csv")
fin = fin[fin["NIK"].isin(target_niks)]

fin_wide = fin.pivot(index="NIK", columns="year",
                      values=["revenue", "net_profit", "total_asset", "total_liability", "operating_cashflow"])
fin_wide.columns = [f"{col}_{year}" for col, year in fin_wide.columns]
fin_wide = fin_wide.reset_index()

fin_wide["revenue_growth_yoy"] = (fin_wide["revenue_2025"] - fin_wide["revenue_2024"]) / fin_wide["revenue_2024"]
fin_wide["net_profit_margin"] = fin_wide["net_profit_2025"] / fin_wide["revenue_2025"]
fin_wide["debt_to_asset_ratio"] = fin_wide["total_liability_2025"] / fin_wide["total_asset_2025"]
fin_wide["operating_cashflow_trend"] = (
    (fin_wide["operating_cashflow_2025"] - fin_wide["operating_cashflow_2024"]) / fin_wide["operating_cashflow_2024"]
)

fin_feat = fin_wide[["NIK", "revenue_growth_yoy", "net_profit_margin",
                      "debt_to_asset_ratio", "operating_cashflow_trend"]]
fin_feat

,NIK,revenue_growth_yoy,net_profit_margin,debt_to_asset_ratio,operating_cashflow_trend
0,3275016008853000,0.187327,0.188213,0.526797,0.490319
1,3276015311842999,0.103461,0.087931,0.624307,0.018347


In [8]:
# --- bank_account: agregasi per NIK ---
bank = pd.read_csv(f"{DATA_DIR}/bank_account.csv")
bank = bank[bank["NIK"].isin(target_niks)]

bank_agg = bank.groupby("NIK").agg(
    average_balance_6m=("average_balance_6m", "mean"),
    average_monthly_credit=("average_monthly_credit", "mean"),
    overdraft_count_6m=("overdraft_count_6m", "sum"),
).reset_index()

bank_agg

,NIK,average_balance_6m,average_monthly_credit,overdraft_count_6m
0,3275016008853000,131687.0,712311.0,0
1,3276015311842999,1655143.0,2003105.0,0


In [9]:
# --- Gabungkan semua jadi satu feature table per applicant ---
base_cols = [
    "application_id", "NIK",
    "legal_entity", "owner_gender", "owner_age", "owner_marital_status", "owner_education",
    "region", "industry", "business_age_year", "employee_count", "monthly_turnover_est",
    "transaction_frequency_monthly", "loan_requested", "collateral_ratio", "collateral_type",
    "certificate_type", "ownership_match", "estimated_dsr", "label",
]
applicant_features = applicant_profile[base_cols].copy()

applicant_features = applicant_features.merge(slik_agg, on="NIK", how="left")
applicant_features = applicant_features.merge(dhn_feat, on="NIK", how="left")
applicant_features = applicant_features.merge(agunan_feat, on="NIK", how="left")
applicant_features = applicant_features.merge(fin_feat, on="NIK", how="left")
applicant_features = applicant_features.merge(bank_agg, on="NIK", how="left")

# NIK tanpa catatan SLIK => tidak ada fasilitas aktif / riwayat buruk yang tercatat
applicant_features["has_credit_history"] = applicant_features["has_credit_history"].fillna(0)
applicant_features["worst_collectability"] = applicant_features["worst_collectability"].fillna(1)
for c in ["n_loans", "total_outstanding", "total_installment"]:
    applicant_features[c] = applicant_features[c].fillna(0)

print("Shape applicant_features:", applicant_features.shape)
assert applicant_features[categorical_cols + numeric_cols].isna().sum().sum() == 0, "Masih ada nilai kosong pada fitur!"
applicant_features[["application_id"] + categorical_cols + numeric_cols]

Shape applicant_features: (2, 36)


,application_id,legal_entity,owner_gender,owner_marital_status,owner_education,region,industry,collateral_type,certificate_type,ownership_match,...,status_dhn_flag,total_collateral_value,land_area_m2,revenue_growth_yoy,net_profit_margin,debt_to_asset_ratio,operating_cashflow_trend,average_balance_6m,average_monthly_credit,overdraft_count_6m
0,APP202602999,UD,P,Belum Menikah,D3,Region 3,Perdagangan,Ruko,SHM,Ya,...,0,12612989000,336.1,0.103461,0.087931,0.624307,0.018347,1655143.0,2003105.0,0
1,APP202603000,UD,P,Belum Menikah,S2,Region 2,Transportasi,Tanah,HGB,Tidak,...,0,8995200000,374.8,0.187327,0.188213,0.526797,0.490319,131687.0,712311.0,0


## 3. Prediksi Kelayakan Kredit

`predict_eligibility()` menerima 1 baris fitur (skema `categorical_cols` +
`numeric_cols`, TANPA `eligibility_score`/identifier), men-transform lewat
`preprocessor` yang sama seperti training, lalu mengembalikan prediksi,
probabilitas, dan **`top_factors`** (kontribusi fitur lokal) — dasar untuk
"Insight/Catatan" di Risk Agent.

In [10]:
def predict_eligibility(features_dict: dict) -> dict:
    row = pd.DataFrame([features_dict])[categorical_cols + numeric_cols]
    row_proc = preprocessor.transform(row)

    proba_positive = float(model.predict_proba(row_proc)[0, 1])
    prediction = positive_class if proba_positive >= 0.5 else "Diterima"

    top_factors = []
    if hasattr(model, "coef_"):
        # Model linear (Logistic Regression): kontribusi = koefisien * nilai fitur (setelah scaling/encoding)
        contrib = pd.Series(model.coef_[0] * row_proc[0], index=feature_names)
        top_factors = [
            {"feature": f, "contribution": round(float(v), 4)}
            for f, v in contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(5).items()
        ]
    elif hasattr(model, "feature_importances_"):
        # Model tree-based (fallback tanpa SHAP): pakai global feature_importances_ sebagai proxy
        importances = pd.Series(model.feature_importances_, index=feature_names)
        top_factors = [
            {"feature": f, "importance": round(float(v), 4)}
            for f, v in importances.sort_values(ascending=False).head(5).items()
        ]

    return {
        "prediction": prediction,
        f"probability_{positive_class.lower()}": round(proba_positive, 4),
        "probability_diterima": round(1 - proba_positive, 4),
        "top_factors": top_factors,
        "model_used": meta["model_name"],
    }

In [11]:
results = []
for _, row in applicant_features.iterrows():
    features_dict = row[categorical_cols + numeric_cols].to_dict()
    pred = predict_eligibility(features_dict)
    pred["application_id"] = row["application_id"]
    pred["actual_label"] = row["label"]
    results.append(pred)

for r in results:
    print("=" * 60)
    print(f"application_id : {r['application_id']}")
    print(f"Label aktual   : {r['actual_label']}")
    print(f"Prediksi model : {r['prediction']}")
    print(json.dumps(
        {k: v for k, v in r.items() if k not in ("application_id", "actual_label", "prediction")},
        indent=2, ensure_ascii=False,
    ))
print("=" * 60)

application_id : APP202602999
Label aktual   : Diterima
Prediksi model : Diterima
{
  "probability_ditolak": 0.0027,
  "probability_diterima": 0.9973,
  "top_factors": [
    {
      "feature": "num__worst_collectability",
      "contribution": -2.4994
    },
    {
      "feature": "num__has_credit_history",
      "contribution": 0.5939
    },
    {
      "feature": "num__status_dhn_flag",
      "contribution": -0.4422
    },
    {
      "feature": "num__total_installment",
      "contribution": 0.3559
    },
    {
      "feature": "cat__ownership_match_Ya",
      "contribution": -0.3256
    }
  ],
  "model_used": "logistic_regression"
}
application_id : APP202603000
Label aktual   : Diterima
Prediksi model : Diterima
{
  "probability_ditolak": 0.0069,
  "probability_diterima": 0.9931,
  "top_factors": [
    {
      "feature": "num__worst_collectability",
      "contribution": -2.4994
    },
    {
      "feature": "num__collateral_ratio",
      "contribution": 0.6131
    },
    {
      

In [12]:
summary = pd.DataFrame([
    {
        "application_id": r["application_id"],
        "actual_label": r["actual_label"],
        "prediction": r["prediction"],
        "match": r["actual_label"] == r["prediction"],
        f"probability_{positive_class.lower()}": r[f"probability_{positive_class.lower()}"],
        "top_factor_1": r["top_factors"][0]["feature"] if r["top_factors"] else None,
    }
    for r in results
])
summary

,application_id,actual_label,prediction,match,probability_ditolak,top_factor_1
0,APP202602999,Diterima,Diterima,True,0.0027,num__worst_collectability
1,APP202603000,Diterima,Diterima,True,0.0069,num__worst_collectability


## Ringkasan

- Data mentah 2 applicant (`APP202602999` tanpa riwayat SLIK, `APP202603000`
  dengan 3 fasilitas SLIK) diambil langsung dari 7 tabel di `data/raw/` dan
  diproses lewat pipeline feature engineering yang identik dengan
  `03_model_training.ipynb`, tanpa menyentuh `eligibility_score` atau kolom
  identifier — konsisten dengan aturan anti-leakage yang sama.
- Prediksi dijalankan memakai artifact model yang **sudah dilatih & disimpan**
  (`model_logistic_regression_20260823_184525.pkl` +
  `preprocessor_20260823_184525.pkl` + meta-nya), bukan melatih ulang — inilah
  pola yang dipakai Risk Agent saat runtime.
- Setiap prediksi disertai `top_factors` (kontribusi fitur lokal) sebagai dasar
  penjelasan keputusan, dan dibandingkan dengan `label` aktual sebagai sanity
  check bahwa pipeline deploy menghasilkan angka yang konsisten dengan hasil
  training.